# 03 - Fases 2+3+4: bucle de aprendizaje activo

REINVENT genera solventes -> BoTorch elige los mas informativos -> Chemprop evalua DG_extract -> el GP se reajusta y premia a REINVENT en la ronda siguiente.

**Add Input:** (1) salida del nb **01**, (2) salida del nb **02**, (3) Dataset con el prior de REINVENT (`reinvent.prior`, del Zenodo del README de REINVENT4), (4) *para reanudar*, la salida de una version previa de ESTE notebook (se detecta sola).

**Primera vez:** `GENERATOR="enumerator"`, `EVALUATOR="surrogate"` para verificar que corre en ~2 min; luego cambia a `reinvent`/`chemprop`.

**Settings:** GPU T4 basta · Internet On · Save & Run All (Commit) corre hasta 12 h sin navegador.

In [ ]:
# ==== Setup comun: codigo en /tmp (efimero), estado en /kaggle/working/alumina (se guarda) ====
import os, glob, json, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/santiagovelcasallas/Data_Science_example_01.git"
BRANCH   = "claude/affectionate-johnson-q9wtj0"   # cambia a "main" tras el merge

ROOT = Path("/kaggle/working/alumina")
CODE = Path("/tmp/repo")
PKG  = CODE / "alumina_pipeline"
for d in ["data", "models", "priors", "runs"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)
os.environ["ALUMINA_ROOT"] = str(ROOT)

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

token = secret("GITHUB_TOKEN")   # solo si el repo es privado
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
if not CODE.exists():
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(CODE)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git clone fallo: " + r.stderr[-300:])
print("Codigo en", PKG)

In [ ]:
# ==== Instalacion (~3-6 min). Se fija la version de torch de Kaggle para no romper CUDA ====
!pip freeze | grep -E "^torch==" > /tmp/constraints.txt; cat /tmp/constraints.txt
!pip install -q -c /tmp/constraints.txt -r {PKG}/kaggle/requirements-kaggle.txt
!pip install -q --no-deps -e {PKG}
!python -c "import torch,rdkit,ase,botorch,alumina_pipeline as a; print('torch',torch.__version__,'| GPU:',torch.cuda.is_available())"

In [ ]:
GENERATOR      = "reinvent"    # reinvent | enumerator
EVALUATOR      = "chemprop"    # chemprop | surrogate
N_ROUNDS       = 10            # TOTAL (al reanudar, incluye las ya hechas)
BATCH          = 8
RL_STEPS       = 50
EXPLICIT_CHECK = False         # chequeo MACE solvente+superficie del top-3 (lento)

In [ ]:
%%bash -s "$GENERATOR"
# REINVENT 4 en venv aislado (solo si GENERATOR == reinvent)
if [ "$1" != "reinvent" ]; then echo "GENERATOR=$1 -> se omite REINVENT"; exit 0; fi
set -e
if [ ! -x /tmp/reinvent-venv/bin/reinvent ]; then
  python -m venv --system-site-packages /tmp/reinvent-venv
  git clone -q --depth 1 https://github.com/MolecularAI/REINVENT4.git /tmp/REINVENT4
  /tmp/reinvent-venv/bin/pip install -q /tmp/REINVENT4 || true
  if ! /tmp/reinvent-venv/bin/reinvent --help >/dev/null 2>&1; then
    cd /tmp/REINVENT4 && /tmp/reinvent-venv/bin/python install.py cu124
  fi
fi
/tmp/reinvent-venv/bin/reinvent --help | head -n 2

In [ ]:
# Copiar inputs: salidas 01/02, prior y (si hay) corrida previa para reanudar
def first(pat, must=None):
    hits = [h for h in sorted(glob.glob("/kaggle/input/**/" + pat + "", recursive=True)) if must is None or must in h]
    return Path(hits[0]) if hits else None

p1, slab = first("phase1_target.json", "/data/"), first("al2o3_slab.extxyz")
if p1 and slab:
    shutil.copy(p1, ROOT/"data/phase1_target.json"); shutil.copy(slab, ROOT/"data/al2o3_slab.extxyz")
    print("ok Fase 1:", p1)
else:
    print("[!] Sin nb 01: la Fase 1 se calcula aqui (minutos en GPU)")

cp = first("chemprop_solvation")
if cp and cp.is_dir():
    shutil.copytree(cp, ROOT/"models/chemprop_solvation", dirs_exist_ok=True); print("ok Chemprop:", cp)
elif EVALUATOR == "chemprop":
    raise FileNotFoundError("Falta nb 02 (chemprop_solvation/) o usa EVALUATOR='surrogate'")

prior = first("*.prior")
if prior:
    shutil.copy(prior, ROOT/"priors/reinvent.prior"); print("ok Prior:", prior)
elif GENERATOR == "reinvent":
    raise FileNotFoundError("Falta reinvent.prior (Add Input) o usa GENERATOR='enumerator'")

RESUME_DIR = None
prev = first("history.csv", "/runs/")
if prev:
    RESUME_DIR = ROOT/"runs"/prev.parent.name
    shutil.copytree(prev.parent, RESUME_DIR, dirs_exist_ok=True); print("ok Reanudando:", RESUME_DIR.name)

In [ ]:
import yaml
yaml.safe_dump({
    "run": {"device": "cuda", "name": "kaggle_loop"},
    "phase1": {"calculator": "mace"},
    "phase2": {"backend": GENERATOR,
               "reinvent": {"executable": "/tmp/reinvent-venv/bin/reinvent",
                            "prior_file": "priors/reinvent.prior", "steps_per_round": RL_STEPS}},
    "phase3": {"backend": EVALUATOR, "explicit_interface": {"enabled": EXPLICIT_CHECK}},
    "phase4": {"batch_size": BATCH},
}, open(ROOT/"overlay_loop.yaml", "w"))
resume_flag = f"--resume {RESUME_DIR}" if RESUME_DIR else ""
print(resume_flag or "Corrida nueva")

In [ ]:
!python -m alumina_pipeline.orchestrator --config {PKG}/configs/pipeline.yaml --overlay {ROOT}/overlay_loop.yaml --rounds {N_ROUNDS} {resume_flag} 2>&1 | grep -v Warning | tee -a {ROOT}/loop.log

In [ ]:
import pandas as pd
run = max((ROOT/"runs").glob("kaggle_loop_*"), key=lambda p: p.stat().st_mtime)
print("Corrida:", run.name)
rounds = pd.DataFrame(json.loads((run/"rounds.json").read_text()))
display(rounds[["round","n_evaluated","n_success","n_lowers_barrier","best_dG_extract_kJ_mol","best_smiles"]])
display(pd.read_csv(run/"top_candidates.csv"))

## Que guardar

Todo queda en `alumina/runs/kaggle_loop_<fecha>/`:

| Archivo | Para que | Imprescindible |
|---|---|---|
| `history.csv` | Cada candidato evaluado, con DG y motivo de fallo | **Si** (es el resultado) |
| `top_candidates.csv` | Los 10 mejores solventes | **Si** |
| `rounds.json` | Progreso y modos de fallo por ronda | **Si** |
| `bo_state.pt` + `reinvent/agent_round*.chkpt` | Estado para **reanudar** otra sesion | **Si**, si continuaras |
| `reinvent/*.csv`, `tb_logs/` | Historial de REINVENT, TensorBoard | Recomendado |

**Para reanudar:** Add Input -> Notebook Output -> esta corrida, sube `N_ROUNDS` y Commit de nuevo.